In [21]:
# Intialization
import os
import sys

os.environ["SPARK_HOME"] = "/home/talentum/spark"
os.environ["PYLIB"] = os.environ["SPARK_HOME"] + "/python/lib"
# In below two lines, use /usr/bin/python2.7 if you want to use Python 2
os.environ["PYSPARK_PYTHON"] = "/usr/bin/python3.6" 
os.environ["PYSPARK_DRIVER_PYTHON"] = "/usr/bin/python3"
sys.path.insert(0, os.environ["PYLIB"] +"/py4j-0.10.7-src.zip")
sys.path.insert(0, os.environ["PYLIB"] +"/pyspark.zip")

# NOTE: Whichever package you want mention here.
# os.environ['PYSPARK_SUBMIT_ARGS'] = '--packages com.databricks:spark-xml_2.11:0.6.0 pyspark-shell' 
# os.environ['PYSPARK_SUBMIT_ARGS'] = '--packages org.apache.spark:spark-avro_2.11:2.4.0 pyspark-shell'
os.environ['PYSPARK_SUBMIT_ARGS'] = '--packages com.databricks:spark-xml_2.11:0.6.0,org.apache.spark:spark-avro_2.11:2.4.3 pyspark-shell'
# os.environ['PYSPARK_SUBMIT_ARGS'] = '--packages com.databricks:spark-xml_2.11:0.6.0,org.apache.spark:spark-avro_2.11:2.4.0 pyspark-shell'

In [22]:
#Entrypoint 2.x
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("Spark SQL basic example").enableHiveSupport().getOrCreate()

# On yarn:
# spark = SparkSession.builder.appName("Spark SQL basic example").enableHiveSupport().master("yarn").getOrCreate()
# specify .master("yarn")

sc = spark.sparkContext

In [20]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

# ============================================================
# Read Silver Cleaned Layer
# ============================================================

goldDF = spark.read.parquet(
    "hdfs://localhost:9000/user/talentum/silver_cleaned"
)

print("Rows Before Cleaning :", goldDF.count())

# ============================================================
# Remove Duplicates
# ============================================================

goldDF = goldDF.dropDuplicates()

# ============================================================
# Remove Impossible Numeric Values
# ============================================================

goldDF = goldDF.filter(col("Applicants_Count") >= 0)

goldDF = goldDF.filter(col("Interview_Rounds").between(1,10))

goldDF = goldDF.filter(col("Number_of_Openings").between(1,100))

goldDF = goldDF.filter(col("Salary_Min") > 0)

goldDF = goldDF.filter(col("Salary_Max") > 0)

goldDF = goldDF.filter(col("Salary_Max") >= col("Salary_Min"))

goldDF = goldDF.filter(col("Selection_Rate").between(0,1))

# ============================================================
# Remove Dummy Dates
# ============================================================

goldDF = goldDF.filter(col("Posted_Date") != lit("2099-01-01"))

goldDF = goldDF.filter(col("Application_Deadline") != lit("2099-01-01"))

# ============================================================
# Replace Invalid String Values
# ============================================================

invalid_values = [
    "",
    " ",
    "\"\"",
    "N/A",
    "n/a",
    "NA",
    "na",
    "null",
    "NULL",
    "None",
    "none",
    "nil",
    "error",
    "ERROR",
    "???",
    "--",
    "unknown",
    "Unknown",
    "undefined",
    "does not exist",
    "#VALUE",
    "void",
    "unset",
    "invalid",
    "0",
    "9999",
    "#####",
    "omitted"
]

string_cols = [
    "Benefits",
    "Company_Name",
    "Currency",
    "Education_Level",
    "Employment_Type",
    "Experience_Level",
    "Industry",
    "Job_Category",
    "Job_Description",
    "Job_Status",
    "Job_Title",
    "Location_City",
    "Location_Country",
    "Location_State",
    "Preferred_Skills",
    "Recruiter_Email",
    "Recruiter_Name",
    "Remote_Option",
    "Required_Skills",
    "Source",
    "Work_Shift"
]

for c in string_cols:

    goldDF = goldDF.withColumn(
        c,
        when(trim(col(c)).isin(invalid_values), None)
        .otherwise(col(c))
    )

# ============================================================
# Fill NULL Strings With Mode
# ============================================================

for c in string_cols:

    row = (
        goldDF
        .filter(col(c).isNotNull())
        .groupBy(c)
        .count()
        .orderBy(desc("count"))
        .first()
    )

    if row:
        goldDF = goldDF.fillna({c: row[0]})

# ============================================================
# Fill Numeric NULL With Median
# ============================================================

numeric_cols = [
    "Applicants_Count",
    "Interview_Rounds",
    "Number_of_Openings",
    "Salary_Min",
    "Salary_Max",
    "Selection_Rate"
]

for c in numeric_cols:

    median = goldDF.approxQuantile(c,[0.5],0.01)[0]

    if median is not None:
        goldDF = goldDF.fillna({c: median})

# ============================================================
# Verify
# ============================================================

print("Rows After Cleaning :", goldDF.count())

print("Duplicate Rows :", goldDF.count() - goldDF.dropDuplicates().count())

goldDF.select([
    sum(when(col(c).isNull(),1).otherwise(0)).alias(c)
    for c in goldDF.columns
]).show(vertical=True)

# ============================================================
# Save Gold Layer (Parquet)
# ============================================================

goldDF.write \
.mode("overwrite") \
.parquet("hdfs://localhost:9000/user/talentum/gold_layer")

# ============================================================
# Save Gold CSV
# ============================================================

goldDF.coalesce(1) \
.write \
.mode("overwrite") \
.option("header","true") \
.csv("hdfs://localhost:9000/user/talentum/gold_csv")

print("Gold Layer Created Successfully")

Rows Before Cleaning : 500000
Rows After Cleaning : 395051
Duplicate Rows : 0
-RECORD 0-------------------
 Applicants_Count     | 0   
 Application_Deadline | 0   
 Benefits             | 0   
 Company_Name         | 0   
 Currency             | 0   
 Education_Level      | 0   
 Employment_Type      | 0   
 Experience_Level     | 0   
 Industry             | 0   
 Interview_Rounds     | 0   
 Job_Category         | 0   
 Job_Description      | 0   
 Job_ID               | 0   
 Job_Status           | 0   
 Job_Title            | 0   
 Location_City        | 0   
 Location_Country     | 0   
 Location_State       | 0   
 Number_of_Openings   | 0   
 Posted_Date          | 0   
 Preferred_Skills     | 0   
 Recruiter_Email      | 0   
 Recruiter_Name       | 0   
 Remote_Option        | 0   
 Required_Skills      | 0   
 Salary_Max           | 0   
 Salary_Min           | 0   
 Selection_Rate       | 0   
 Source               | 0   
 Work_Shift           | 0   

Gold Layer Created Suc

In [25]:
goldDF = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv("hdfs://localhost:9000/user/talentum/gold_csv")

goldDF.show(20, truncate=False)

+----------------+--------------------+------------------------------------------------------------------------------------------------------------------------------------------+----------------+--------+-------------------------------------+---------------+----------------+---------------------+----------------+---------------------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+---------+----------+-------------------------+-------------+----------------+----------------+------------------+-------------------+-------------------------------------------------------+---------------------------+----------------+-------------+-------

+-------+-----------------+-----------------+-----------------+-----------------+------------------+------------------+------------------+------------------+-----------------+-----------------+--------------------+---------+-----------------+-----------------+------------------+------------------+------------------+------------------+------------------+------------------+-----------------+------------------+-----------------+------------------+-----------------+------------------+------------------+-----------------+
|summary| Applicants_Count|         Benefits|     Company_Name|         Currency|   Education_Level|   Employment_Type|  Experience_Level|          Industry| Interview_Rounds|     Job_Category|     Job_Description|   Job_ID|       Job_Status|        Job_Title|     Location_City|  Location_Country|    Location_State|Number_of_Openings|  Preferred_Skills|   Recruiter_Email|   Recruiter_Name|     Remote_Option|  Required_Skills|        Salary_Max|       Salary_Min|    Selecti

In [19]:
goldDF = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv("file:///home/talentum/Downloads/gold_csv/Job_Market_Gold.csv")

goldDF.show(20, truncate=False)

+----------------+--------------------+------------------------------------------------------------------------------------------------------------------------------------------+-------------+--------+-------------------------------------+---------------+----------------+---------------------+----------------+---------------------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+---------+----------+-------------------------+-------------+----------------+--------------+------------------+-------------------+--------------------------------+--------------------------+------------------+-------------+----------------------------------